In [1]:
import torch 
import torch.nn as nn
from torch.nn import functional
import re

In [2]:
from datasets import load_dataset

dataset = load_dataset("roneneldan/TinyStories" , split='train[:50000]')

/Users/mac/miniconda3/envs/learning_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [25]:
with open("tinystories.txt", "w", encoding="utf-8") as f:
    for story in dataset["text"]:
        f.write(story)
        f.write("\n\n")

In [76]:
with open('tinystories.txt' , 'r' , encoding='utf-8') as r:
    text = r.read()
print(text[:1000])

One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."

Together, they shared the needle and sewed the button on Lily's shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.

Once upon a time, there was a little car named Beep. Beep loved to go fast and play in the sun. Beep was a healthy car because he always had good fuel. Good fuel made Beep happy and strong.

One day, Beep was driving in the park when he saw a big tree. The tree had many leaves that were falling. 

In [4]:
words = text.split()
words[:10]

['One', 'day,', 'a', 'little', 'girl', 'named', 'Lily', 'found', 'a', 'needle']

In [5]:
# We will make our own BPE tokenizer:
# char-level-Bigram Model

chars = sorted(list(set(text)))
v = len(chars)

# Own BPE tokenizer:-

In [ ]:
from collections import defaultdict

sub_words = 3000
vocab_merge = {}

word_freqs = defaultdict(int)
for word in words:
    tokens = tuple(list(word) + ['</w>'])
    word_freqs[tokens] += 1

for s in range(sub_words):
    # 2. Count bigram frequencies across unique words only
    bigram_counts = defaultdict(int)
    for tokens, freq in word_freqs.items():
        for i in range(len(tokens) - 1):
            bigram = (tokens[i], tokens[i+1]) # -> adjecent pairs store ho rhae
            bigram_counts[bigram] += freq

    if not bigram_counts:
        break

    # 3. Find the most frequent bigram
    best_bigram = max(bigram_counts, key=bigram_counts.get)
    new_token = "".join(best_bigram)
    vocab_merge[best_bigram] = new_token

    # 4. Merge bigram in word_freqs
    new_word_freqs = defaultdict(int)
    for tokens, freq in word_freqs.items():
        new_tokens = []
        i = 0
        while i < len(tokens):
            if i < len(tokens) - 1 and (tokens[i], tokens[i+1]) == best_bigram:
                new_tokens.append(new_token)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        new_word_freqs[tuple(new_tokens)] += freq

    word_freqs = new_word_freqs # -> har word kitne baar ayaa hai?

In [25]:
words[:10]

['One', 'day,', 'a', 'little', 'girl', 'named', 'Lily', 'found', 'a', 'needle']

# Making Function of encode():-

To process words:

In [33]:
def encode_word(word, vocab_merge):
    tokens = list(word) + ["</w>"]

    while True:
        pairs = [
            (tokens[i], tokens[i + 1]) for i in range(len(tokens) - 1)
        ]

        best_pair = None
        for p in pairs:
            if p in vocab_merge:
                best_pair = p 
                break

        if best_pair is None:
            break

        new_token = vocab_merge[best_pair]
        new_tokens = []
        i = 0
        while i < len(tokens):
            if (
                i < len(tokens) - 1
                and (tokens[i], tokens[i + 1]) == best_pair
            ):
                new_tokens.append(new_token)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1

        tokens = new_tokens

    return tokens

To process text:

In [50]:
def encode_text(text , vocab_merge , vocab):
    words = text.split()
    
    ids = []

    for w in words:
        tokens = (encode_word(w , vocab_merge))

        for i in tokens:
            ids.append(vocab[i])

    return ids

# BPE tokens → integer token IDs

In [46]:
vocab = {}
for w in words:
    for token in list(w) + ["</w>"]:
        if token not in vocab:
            vocab[token] = len(vocab)
            
for token in vocab_merge.values():
    if token not in vocab:
        vocab[token] = len(vocab)

In [57]:
text = "Oneday"

encode_text(text , vocab_merge , vocab)

[184, 618, 5, 107]

In [58]:
tokens = []

for word in text.split():
    tokens.extend(encode_word(word, vocab_merge))

print(tokens)

['On', 'ed', 'a', 'y</w>']


In [59]:
tokens = []

for word in text.split():
    tokens.extend(encode_word(word, vocab_merge))

ids = [vocab[token] for token in tokens]

for token, idx in zip(tokens, ids):
    print(token, "->", idx)

On -> 184
ed -> 618
a -> 5
y</w> -> 107


In [68]:
def decode(ids , vocab):
    ids_token = {idx:token for token , idx in vocab.items()}
    tokens = [ids_token[idx] for idx in ids]
    text = "".join(tokens)
    return text.strip().replace("</w>" , " ")

In [70]:
text = "One day"
ids = encode_text(text, vocab_merge, vocab)

decoded = decode(ids, vocab)
print("Original:", text)
print("IDs:", ids)
print("Decoded:", decoded)

Original: One day
IDs: [259, 2704, 3]
Decoded: One day 


In [85]:
len(text)
# len(words)


44657053

In [71]:
import torch
import torch.nn as nn
from torch.nn import functional as F
from collections import defaultdict


with open('tinystories.txt' , 'r' , encoding='utf-8') as f:
    text = f.read()

words = text.split()

unique_chars = sorted(list(set(text)))
v = len(unique_chars)

vocab_merges = {}
sub_words = 3000

word_freq = defaultdict(int)
for w in words:
    token = tuple(list(w) + ['</w>'])
    word_freq[token] += 1


for i in range(sub_words):
    bigram_counts = defaultdict(int)
    for token , freq in word_freq.items():
        for t in range(len(token) - 1):
            bigram = (token[t] , token[t+1])
            bigram_counts[bigram] += freq

    if not bigram_counts:
        break

    best_bigram = max(bigram_counts , key=bigram_counts.get)
    new_token = "".join(best_bigram)
    vocab_merges[best_bigram] = new_token

    new_word_freq = defaultdict(int)
    for tokens , freq in word_freq.items():
        new_tokens = []
        j = 0
        while j < len(tokens):
            if j < len(tokens) - 1 and (tokens[j] , tokens[j+1]) == best_bigram:
                new_tokens.append(new_token)
                j += 2
            else:
                new_tokens.append(tokens[j])
                j += 1
        new_word_freq[tuple(new_tokens)] += freq

    word_freq = new_word_freq 


def encode_word(word, vocab_merge):
    tokens = list(word) + ["</w>"]

    merge_ranks = {pair: rank for rank, pair in enumerate(vocab_merge.keys())}
    while len(tokens) > 1:
        pairs = [(tokens[i], tokens[i + 1]) for i in range(len(tokens) - 1)]
        valid_pairs = [pair for pair in pairs if pair in merge_ranks]
        if not valid_pairs:
            break
        best_pair = min(valid_pairs , key=lambda pair: merge_ranks[pair])

        new_token = vocab_merge[best_pair]
        new_tokens = []

        i = 0
        while i < len(tokens):
            if (i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == best_pair):
                new_tokens.append(new_token)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1

        tokens = new_tokens
    return tokens


vocab = {}
for char in unique_chars:
    if char not in vocab:
        vocab[char] = len(vocab)

if '</w>' not in vocab:
    vocab['</w>'] = len(vocab)

for token in vocab_merges.values():
    if token not in vocab:
        vocab[token] = len(vocab)


def encode_text(text , vocab_merge , vocab):
    words = text.split()

    ids = []
    for w in words:
        tokens = encode_word(w , vocab_merge)
        for i in tokens:
            ids.append(vocab[i])

    return ids


def decode(idx , vocab):
    ids_token = {idx_val: token for token , idx_val in vocab.items()}
    tokens = [ids_token[id] for id in idx]
    text_out = "".join(tokens)
    return text_out.strip().replace("</w>" , " ")


# Example
text = "One day"
ids = encode_text(text, vocab_merges, vocab)

decoded = decode(ids, vocab)
print("Original:", text)
print("IDs:", ids)
print("Decoded:", decoded)

Original: One day
IDs: [263, 353]
Decoded: One day 


In [80]:
len(text)

44657053

In [78]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

In [94]:
torch.manual_seed(1337)
batch_size = 32
block_size = 64
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(0 , len(data) - block_size , (batch_size ,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x , y

In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F
from collections import defaultdict


if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")


with open('tinystories.txt' , 'r' , encoding='utf-8') as f:
    text = f.read()

words = text.split()

unique_chars = sorted(list(set(text)))

# BPE:
vocab_merges = {}
sub_words = 1000

word_freq = defaultdict(int)
for w in words:
    token = tuple(list(w) + ['</w>'])
    word_freq[token] += 1

for i in range(sub_words):
    bigram_counts = defaultdict(int)
    for token , freq in word_freq.items():
        for t in range(len(token) - 1):
            bigram = (token[t] , token[t+1])
            bigram_counts[bigram] += freq

    if not bigram_counts:
        break

    best_bigram = max(bigram_counts , key=bigram_counts.get)
    new_token = "".join(best_bigram)
    vocab_merges[best_bigram] = new_token

    new_word_freq = defaultdict(int)
    for tokens , freq in word_freq.items():
        new_tokens = []
        j = 0
        while j < len(tokens):
            if j < len(tokens) - 1 and (tokens[j] , tokens[j+1]) == best_bigram:
                new_tokens.append(new_token)
                j += 2
            else:
                new_tokens.append(tokens[j])
                j += 1
        new_word_freq[tuple(new_tokens)] += freq

    word_freq = new_word_freq 

# encode_word
merge_ranks = {pair: rank for rank, pair in enumerate(vocab_merges.keys())}
def encode_word(word, vocab_merges , merge_ranks):
    tokens = list(word) + ["</w>"]

    while len(tokens) > 1:
        pairs = [(tokens[i], tokens[i + 1]) for i in range(len(tokens) - 1)]
        valid_pairs = [pair for pair in pairs if pair in merge_ranks]
        if not valid_pairs:
            break
        best_pair = min(valid_pairs , key=lambda pair: merge_ranks[pair])

        new_token = vocab_merges[best_pair]
        new_tokens = []

        i = 0
        while i < len(tokens):
            if (i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == best_pair):
                new_tokens.append(new_token)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1

        tokens = new_tokens
    return tokens

# vocab
vocab = {}
for char in unique_chars:
    if char not in vocab:
        vocab[char] = len(vocab)

if '</w>' not in vocab:
    vocab['</w>'] = len(vocab)

for token in vocab_merges.values():
    if token not in vocab:
        vocab[token] = len(vocab)

# encode_text
# Solution of bug: (Maintaining a chacing dict)
def encode_text(text , vocab , vocab_merges , merge_ranks):
    words = text.split()
    cache_words = {} # -> memory saving dict (solution)
    ids = []

    for w in words:
        if w not in cache_words:
            tokens = encode_word(w , vocab_merges ,  merge_ranks)
            cache_words[w] = [vocab[t] for t in tokens]
        ids.extend(cache_words[w])

    return ids

# decode:
def decode(idx , vocab):
    ids_token = {idx_val: token for token , idx_val in vocab.items()}
    tokens = [ids_token[id] for id in idx]
    text_out = "".join(tokens)
    return text_out.strip().replace("</w>" , " ")


# Making data (train , val)
data = encode_text(text , vocab ,vocab_merges , merge_ranks)

n = int(0.9*len(data))
train_data = torch.tensor(data[:n] , dtype=torch.long).to(device)
val_data = torch.tensor(data[n:] , dtype=torch.long).to(device)

# get_batch
torch.manual_seed(1337)
batch_size = 32
block_size = 64
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(0 , len(data) - block_size , (batch_size ,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x , y = x.to(device) , y.to(device)
    return x , y

In [3]:
xb , yb = get_batch('train')
print(f'xb: {xb.shape}')
print(f'yb: {yb.shape}')

xb: torch.Size([32, 64])
yb: torch.Size([32, 64])


In [7]:
vocab_size = len(vocab)
torch.manual_seed(1337)
num_emb = 256

class BLM(nn.Module):
    def __init__(self , vocab_size):
        super().__init__()
        self.embedding_table = nn.Embedding(vocab_size , num_emb)
        self.positional_table = nn.Embedding(block_size , num_emb)
# LM Head: Dense Vector ko wapas Vocab size logits mein map karne ke liye
        self.lm_head = nn.Linear(num_emb , vocab_size)

    def forward(self , input_token , target=None):
        B,T = input_token.shape

        tok_emb = self.embedding_table(input_token)
        position_emb = torch.arange(T , device= input_token.device)
        pos_emb = self.positional_table(position_emb)

        combined_table = pos_emb + tok_emb

        logits = self.lm_head(combined_table)

        if target is None:
            loss = None
        else:
            B,T,C = logits.shape

            logits = logits.view(B*T , C)
            target = target.view(B*T)

            loss = F.cross_entropy(logits , target)
        return logits , loss

In [ ]:
class singleHead(nn.Module):
    def __init__(self , head_size):
        super().__init__()
        self.key = nn.Linear(num_emb , head_size , bias=False)
        self.value = nn.Linear(num_emb , head_size , bias=False)
        self.query = nn.Linear(num_emb , head_size ,bias=False)
        self.register_buffer('tril' , torch.tril(torch.ones(block_size , block_size)))


    def forward(self , input):
        # input: (B,T,C)
        # output: (B,T,head_size)
        B,T,C = input.shape
        k = self.key(input)
        q = self.query(input)
        v = self.value(input)

        wei = k @ q.transpose(-2,-1)*k.shape[-1]**-0.5 # -> (B, T, head_size) @ (B, head_size, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei , dim=-1)
        output = wei @ v

        return output

$$\text{head\_size} = \frac{\text{num\_emb}}{\text{num\_heads}} = \frac{256}{4} = 64$$

$$\text{Head}_i = \text{softmax}\left(\frac{Q_i \times K_i^T}{\sqrt{\text{head\_size}}} + \text{Mask}\right) \times V_i$$

In [15]:
# Input x ──► [Linear Layers] ──► Q, K, V ──► [Q @ K^T / √d] ──► Masking & Softmax ──► [@ V] ──► Output
class multiHead(nn.Module):
    def __init__(self , head_size , num_head):
        super().__init__()
        self.heads = nn.ModuleList([singleHead(head_size) for _ in range(num_head)])
# self.proj in saare outputs ke features ko mix/blend karke original embedding dimension mein optimize karta hai.
        self.proj = nn.Linear(head_size*num_head , head_size*num_head)

    def forward(self , input):
        output = [head(input) for head in self.heads]
        output = torch.cat(output , dim = -1)
        out = self.proj(output)
        return output

In [16]:
class feedForward(nn.Module):
    def __init__(self , num_emb):
        super().__init__()
        self.net = nn.Sequential(
        nn.Linear(num_emb , 4*num_emb),
        nn.ReLU(),
        nn.Linear(4*num_emb , num_emb))

    def forward(self , x):
        out = self.net(x)
        return out

In [18]:
class Block(nn.Module):
    def __init__(self , num_emb , num_head):
        super().__init__()
        head_size = num_emb // num_head

        self.FFN = feedForward(num_emb)
        self.MH = multiHead(head_size , num_head)

        self.layer1 = nn.LayerNorm(num_emb)
        self.layer2 = nn.LayerNorm(num_emb)

    def forward(self , x):
        x = self.MH(self.layer1(x))
        x = self.FFN(self.layer2(x))
        return x

In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F
from collections import defaultdict

torch.manual_seed(1337)
batch_size = 32
block_size = 64


if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")


with open('tinystories.txt' , 'r' , encoding='utf-8') as f:
    text = f.read()

words = text.split()

unique_chars = sorted(list(set(text)))

# BPE:
vocab_merges = {}
sub_words = 1000

word_freq = defaultdict(int)
for w in words:
    token = tuple(list(w) + ['</w>'])
    word_freq[token] += 1

for i in range(sub_words):
    bigram_counts = defaultdict(int)
    for token , freq in word_freq.items():
        for t in range(len(token) - 1):
            bigram = (token[t] , token[t+1])
            bigram_counts[bigram] += freq

    if not bigram_counts:
        break

    best_bigram = max(bigram_counts , key=bigram_counts.get)
    new_token = "".join(best_bigram)
    vocab_merges[best_bigram] = new_token

    new_word_freq = defaultdict(int)
    for tokens , freq in word_freq.items():
        new_tokens = []
        j = 0
        while j < len(tokens):
            if j < len(tokens) - 1 and (tokens[j] , tokens[j+1]) == best_bigram:
                new_tokens.append(new_token)
                j += 2
            else:
                new_tokens.append(tokens[j])
                j += 1
        new_word_freq[tuple(new_tokens)] += freq

    word_freq = new_word_freq 

# encode_word
merge_ranks = {pair: rank for rank, pair in enumerate(vocab_merges.keys())}
def encode_word(word, vocab_merges , merge_ranks):
    tokens = list(word) + ["</w>"]

    while len(tokens) > 1:
        pairs = [(tokens[i], tokens[i + 1]) for i in range(len(tokens) - 1)]
        valid_pairs = [pair for pair in pairs if pair in merge_ranks]
        if not valid_pairs:
            break
        best_pair = min(valid_pairs , key=lambda pair: merge_ranks[pair])

        new_token = vocab_merges[best_pair]
        new_tokens = []

        i = 0
        while i < len(tokens):
            if (i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == best_pair):
                new_tokens.append(new_token)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1

        tokens = new_tokens
    return tokens

# vocab
vocab = {}
for char in unique_chars:
    if char not in vocab:
        vocab[char] = len(vocab)

if '</w>' not in vocab:
    vocab['</w>'] = len(vocab)

for token in vocab_merges.values():
    if token not in vocab:
        vocab[token] = len(vocab)

# encode_text
# Solution of bug: (Maintaining a chacing dict)
def encode_text(text , vocab , vocab_merges , merge_ranks):
    words = text.split()
    cache_words = {} # -> memory saving dict (solution)
    ids = []

    for w in words:
        if w not in cache_words:
            tokens = encode_word(w , vocab_merges ,  merge_ranks)
            cache_words[w] = [vocab[t] for t in tokens]
        ids.extend(cache_words[w])

    return ids

# decode:
def decode(idx , vocab):
    ids_token = {idx_val: token for token , idx_val in vocab.items()}
    tokens = [ids_token[id] for id in idx]
    text_out = "".join(tokens)
    return text_out.strip().replace("</w>" , " ")


# Making data (train , val)
data = encode_text(text , vocab ,vocab_merges , merge_ranks)

n = int(0.9*len(data))
train_data = torch.tensor(data[:n] , dtype=torch.long).to(device)
val_data = torch.tensor(data[n:] , dtype=torch.long).to(device)

# get_batch
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(0 , len(data) - block_size , (batch_size ,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x , y = x.to(device) , y.to(device)
    return x , y

vocab_size = len(vocab)
num_emb = 256

class BLM(nn.Module):
    def __init__(self , vocab_size , num_head =4 , n_layer= 4):
        super().__init__()
        self.embedding_table = nn.Embedding(vocab_size ,num_emb)
        self.positional_table = nn.Embedding(block_size , num_emb)

        self.block = nn.Sequential(*[Block(num_emb , num_head) for _ in range(n_layer)]) # Fix of bug
        self.layernorm = nn.LayerNorm(num_emb) # Fix of bug
        self.lm_head = nn.Linear(num_emb , vocab_size)

    def forward(self , input_tokens , target=None):
        B,T = input_tokens.shape
        tok_emb = self.embedding_table(input_tokens)
        position = torch.arange(T , device= input_tokens.device)
        pos_emb = self.positional_table(position)
        x = tok_emb + pos_emb
        x = self.block(x)
        x = self.layernorm(x)
        logits = self.lm_head(x)

        if target is None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T , C)
            target = target.view(B*T)
            loss = F.cross_entropy(logits , target)
        return logits , loss

    def generate(self , idx , max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cont = idx[: , -block_size:] # -> bringing last 8 tokens
            logits,_ = self(idx_cont)
            logits = logits[: , -1 , :]
            probs = F.softmax(logits , dim=-1)
            idx_next = torch.multinomial(probs , num_samples=1)
            idx = torch.cat((idx , idx_next) , dim=-1)
        return idx

class singleHead(nn.Module):
    def __init__(self , head_size):
        super().__init__()
        self.key = nn.Linear(num_emb , head_size , bias=False)
        self.value = nn.Linear(num_emb , head_size , bias=False)
        self.query = nn.Linear(num_emb , head_size , bias=False)
        self.register_buffer('tril' , torch.tril(torch.ones(block_size , block_size)))

    def forward(self , input):
        B,T,C = input.shape
        k = self.key(input)
        q = self.query(input)
        v = self.value(input)

        wei = q @ k.transpose(-2,-1) * k.shape[-1] ** -0.5
        wei = wei.masked_fill(self.tril[:T , :T] == 0 ,float('-inf') )
        wei = F.softmax(wei , dim=-1)

        output = wei @ v

        return output

class multiHead(nn.Module):
    def __init__(self , head_size , num_head):
        super().__init__()
        self.heads = (singleHead(head_size) for _ in range(num_head))
        self.porj = nn.Linear(head_size*num_head , head_size*num_head)
    def forward(self , input):
        output = [head(input) for head in self.heads]
        output = torch.cat(output , dim = -1)
        out = self.proj(output)
        return output 

class feedForward(nn.Module):
    def __init__(self , num_emb):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(num_emb , 4*num_emb),
            nn.ReLU(),
            nn.Linear(4*num_emb , num_emb)
        )

    def forward(self , x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self , num_emb , num_head):
        super().__init__()
        head_size = num_emb // num_head

        self.FNN = feedForward(num_emb)
        self.MH = multiHead(head_size , num_head)
        self.layer1 = nn.LayerNorm(num_emb)
        self.layer2 = nn.LayerNorm(num_emb)

    def forward(self , x):
        x = self.MH(self.layer1(x)) + x # -> Fix of bug
        x = self.FNN(self.layer2(x)) + x
        return x



model = BLM(vocab_size).to(device)
# Optimizer
optimizer = torch.optim.AdamW(model.parameters() , lr = 1e-3)


max_loop = 5000
eval_interval = 500

for iter in range(max_loop):
    if iter % eval_interval == 0:
         xb , yb = get_batch('val')
         _ , loss = model(xb,yb)
         print(f'step {iter}: val loss {loss.item():.4f}')


    xb , yb = get_batch('train')
    logits , loss = model(xb,yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()


context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated_ids = model.generate(context, max_new_tokens=500)[0].tolist()
print(decode(generated_ids, vocab))


RuntimeError: Tensor for argument weight is on cpu but expected on mps